# 텍스트 분할(Text Splitter) : Chunking
LangChain의 텍스트 분할(Text Splitter) 은 긴 문서(예: PDF, 웹페이지, 텍스트 파일 등)를 벡터 임베딩(Vector Embedding) 단계 전에 <br>
적절한 크기의 조각(Chunk) 으로 나누기 위한 핵심 도구이다. <br>
RAG(Retrieval-Augmented Generation) 파이프라인의 성능은 이 “문서 분할 전략”에 크게 좌우된다. <br>

In [ ]:
!pip install langchain-experimental

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")

True

### [1] CharacterTextSplitter
: CharacterTextSplitter 는 텍스트를 문자(Character) 단위로 단순하게 자르는 기본 분할기(Text Splitter) 이다. <br>
이 클래스는 문장을 “특정 구분자(separator)” 기준으로 쪼개고, 각 조각(chunk)을 지정된 크기(chunk_size)로 묶는 방식으로 동작한다.

In [2]:
from langchain_text_splitters import CharacterTextSplitter

text = """
LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.
문서 로드, 텍스트 분할, 임베딩, 검색, 체인 등의 기능을 제공합니다.
CharacterTextSplitter 는 텍스트를 문자(Character) 단위로 단순하게 자르는 기본 분할기(Text Splitter) 이다.
이 클래스는 문장을 “특정 구분자(separator)” 기준으로 쪼개고, 각 조각(chunk)을 지정된 크기(chunk_size)로 묶는 방식으로 동작한다
"""

# Character 단위 분할기 생성
splitter = CharacterTextSplitter(
    separator="\n",      # 줄바꿈 기준으로 분할
    chunk_size=40,       # 각 청크 최대 40자
    chunk_overlap=10     # 앞뒤 청크 10자 겹치기
)

# 분할 실행
chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"청크 {i+1}:", chunk)

# Created a chunk of size 81, which is longer than the specified 40
# LangChain의 CharacterTextSplitter 동작 방식의 핵심 특징 :
# “chunk_size는 엄격한 자르기 기준이 아니라 목표 크기(target)”이기 때문이다.
# 만약 하나의 구분자 사이 텍스트가 이미 chunk_size보다 길면,
# LangChain은 그 문장을 자르지 않고 그대로 하나의 청크로 유지한다

Created a chunk of size 81, which is longer than the specified 40


청크 1: LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.
청크 2: 문서 로드, 텍스트 분할, 임베딩, 검색, 체인 등의 기능을 제공합니다.
청크 3: CharacterTextSplitter 는 텍스트를 문자(Character) 단위로 단순하게 자르는 기본 분할기(Text Splitter) 이다.
청크 4: 이 클래스는 문장을 “특정 구분자(separator)” 기준으로 쪼개고, 각 조각(chunk)을 지정된 크기(chunk_size)로 묶는 방식으로 동작한다


In [3]:
# 문서를 개별 문자를 단위로 나누기 
from langchain_text_splitters import CharacterTextSplitter

text = """
LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.
문서 로드, 텍스트 분할, 임베딩, 검색, 체인 등의 기능을 제공합니다.
CharacterTextSplitter 는 텍스트를 문자(Character) 단위로 단순하게 자르는 기본 분할기(Text Splitter) 이다.
이 클래스는 문장을 “특정 구분자(separator)” 기준으로 쪼개고, 각 조각(chunk)을 지정된 크기(chunk_size)로 묶는 방식으로 동작한다
"""

# Character 단위 분할기 생성
splitter = CharacterTextSplitter(
    separator="",      # 문서를 개별 문자를 단위로 나누기 
    chunk_size=40,       # 각 청크 최대 40자
    chunk_overlap=10     # 앞뒤 청크 10자 겹치기
)

# 분할 실행
chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"청크 {i+1}:", chunk)

청크 1: LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.
청크 2: 프레임워크입니다.
문서 로드, 텍스트 분할, 임베딩, 검색, 체인 등의
청크 3: 검색, 체인 등의 기능을 제공합니다.
CharacterTextSplitt
청크 4: TextSplitter 는 텍스트를 문자(Character) 단위로 단순
청크 5: er) 단위로 단순하게 자르는 기본 분할기(Text Splitter) 이
청크 6: plitter) 이다.
이 클래스는 문장을 “특정 구분자(separato
청크 7: 자(separator)” 기준으로 쪼개고, 각 조각(chunk)을 지정된
청크 8: hunk)을 지정된 크기(chunk_size)로 묶는 방식으로 동작한다


### [2] RecursiveCharacterTextSplitter
: RecursiveCharacterTextSplitter 는 LangChain에서 가장 지능적인 텍스트 분할기(Text Splitter) 이다 <br>
긴 문서를 문맥 손실 없이, LLM 입력 제한(token limit) 에 맞게 잘게 나누기 위해 설계된 클래스이다. <br>
가장 많이 쓰이고, RAG 파이프라인의 핵심. <br>

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.
문서 로드, 텍스트 분할, 임베딩, 검색, 체인 등의 기능을 제공합니다.
LangChain의 핵심 구성 요소는 체인, 메모리, 에이전트입니다.
RecursiveCharacterTextSplitter 는 LangChain에서 가장 지능적인 텍스트 분할기(Text Splitter) 이다
긴 문서를 문맥 손실 없이, LLM 입력 제한(token limit) 에 맞게 잘게 나누기 위해 설계된 클래스이다.
가장 많이 쓰이고, RAG 파이프라인의 핵심.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=40,        # 청크 최대 40자
    chunk_overlap=10,     # 앞뒤로 10자씩 겹침

    # 문자열 길이를 계산하는 함수를 지정합니다.
    length_function=len,   
    # 구분자로 정규식을 사용할지 여부를 설정합니다.
    is_separator_regex=False,
)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"청크 {i+1}: {chunk}")


청크 1: LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.
청크 2: 문서 로드, 텍스트 분할, 임베딩, 검색, 체인 등의 기능을
청크 3: 체인 등의 기능을 제공합니다.
청크 4: LangChain의 핵심 구성 요소는 체인, 메모리, 에이전트입니다.
청크 5: RecursiveCharacterTextSplitter 는
청크 6: 는 LangChain에서 가장 지능적인 텍스트 분할기(Text
청크 7: 분할기(Text Splitter) 이다
청크 8: 긴 문서를 문맥 손실 없이, LLM 입력 제한(token limit)
청크 9: limit) 에 맞게 잘게 나누기 위해 설계된 클래스이다.
청크 10: 가장 많이 쓰이고, RAG 파이프라인의 핵심.


### [3] TokenTextSplitter
: TokenTextSplitter는 텍스트를 문자 수가 아니라 토큰 단위로 나누는 클래스이다. <br>
모델의 토큰 한도(예: 8192 tokens)를 초과하지 않게 문서를 나누고, 모델의 비용/속도/성능 최적화에 유용하다.

In [5]:
from langchain_text_splitters import TokenTextSplitter

text = """
LangChain은 LLM 애플리케이션 개발을 단순화하는 프레임워크입니다.
문서 로딩, 텍스트 분할, 임베딩, 검색, 체인 구성 등의 기능을 제공합니다.
TokenTextSplitter는 텍스트를 문자 수가 아니라 토큰 단위로 나누는 클래스이다.
모델의 토큰 한도를 초과하지 않게 문서를 나누고, 모델의 비용/속도/성능 최적화에 유용하다.
"""

splitter = TokenTextSplitter(
    chunk_size=80,       # 최대 80토큰
    chunk_overlap=10     # 앞뒤 10토큰 겹치기
)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"청크 {i+1}: {chunk}\n")

# 출력된 � 기호는 청크를 자르는 과정에서 한글이 두 바이트 
# 이상으로 인코딩된 상태에서 중간이 잘려버린 현상 때문이다.
# 해결방법은 -> RecursiveCharacterTextSplitter 사용

청크 1: 
LangChain은 LLM 애플리케이션 개발을 단순화하는 프레임워크입니다.
문서 �

청크 2: �.
문서 로딩, 텍스트 분할, 임베딩, 검색, 체인 구성 등의 기능을 제�

청크 3: �능을 제공합니다.
TokenTextSplitter는 텍스트를 문자 수가 아니라 토큰 단위로 나�

청크 4: 위로 나누는 클래스이다.
모델의 토큰 한도를 초과하지 않게 문서�

청크 5: �� 문서를 나누고, 모델의 비용/속도/성능 최적화에 유용하다.




## Chunking 전략 비교 : 고정(Fixed) vs 시맨틱(Semantic) vs 계층적(Hierarchical)
문서를 어떤 기준으로 자르느냐에 따라 검색 정확도와 처리 비용이 크게 달라진다. <br>
아래 세 가지 전략은 실무에서 가장 많이 비교되는 Chunking 방식이다. <br>

| 구분 | 고정 Chunking (Fixed) | 시맨틱 Chunking (Semantic) | 계층적 Chunking (Hierarchical) |
| --- | --- | --- | --- |
| 분할 기준 | 글자 수 또는 토큰 수 | 문장의 의미 | 문서 구조(장→절→문단) |
| 구현 난이도 | 매우 쉬움 | 보통~어려움 | 보통 |
| 검색 정확도 | 낮음~보통 | 높음 | 매우 높음 |
| 처리 속도 | 매우 빠름 | 느림 | 보통 |
| 문맥 유지 | 낮음 | 높음 | 매우 높음 |
| 구현 비용 | 낮음 | 높음 | 보통 |
| 추천도 | ★★★☆☆ | ★★★★☆ | ★★★★★ |

### [4] 고정 Chunking (Fixed Chunking)
: 글자 수 또는 토큰 수를 기준으로 문서를 일정한 크기로 자르는 방식이다. <br>
앞서 실습한 CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter 가 모두 고정 Chunking에 해당한다. <br>
구현이 가장 쉽고 속도도 빠르지만, 문장이나 문단 중간에서 잘릴 수 있어 문맥이 끊기고 검색 정확도가 떨어질 수 있다.

In [6]:
# 고정(Fixed) Chunking 예제 : 글자 수 기준으로 균일하게 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
Chunking 전략은 RAG 파이프라인의 검색 성능을 좌우하는 핵심 요소이다.
고정 Chunking은 문서의 의미와 상관없이 정해진 글자 수 또는 토큰 수로 문서를 자른다.
구현이 단순하고 처리 속도가 빠르다는 장점이 있지만, 문장 중간이 잘려 문맥이 끊길 위험이 있다.
"""

fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=40,      # 글자 수 기준 고정 크기
    chunk_overlap=5
)

fixed_chunks = fixed_splitter.split_text(text)

for i, chunk in enumerate(fixed_chunks):
    print(f"[고정] 청크 {i+1}:", chunk)

[고정] 청크 1: Chunking 전략은 RAG 파이프라인의 검색 성능을 좌우하는 핵심
[고정] 청크 2: 핵심 요소이다.
[고정] 청크 3: 고정 Chunking은 문서의 의미와 상관없이 정해진 글자 수 또는
[고정] 청크 4: 수 또는 토큰 수로 문서를 자른다.
[고정] 청크 5: 구현이 단순하고 처리 속도가 빠르다는 장점이 있지만, 문장 중간이 잘려
[고정] 청크 6: 잘려 문맥이 끊길 위험이 있다.


### [5] 시맨틱 Chunking (Semantic Chunking)
: 문장 단위 임베딩을 계산한 뒤, 인접 문장 간 의미 유사도가 크게 떨어지는 지점(breakpoint)을 기준으로 분할하는 방식이다. <br>
langchain_experimental 의 SemanticChunker 를 사용하며, 임베딩 모델(OpenAIEmbeddings 등)이 필요하다. <br>
문맥이 유지된 채로 청크가 구성되어 검색 정확도가 높지만, 문장마다 임베딩을 계산해야 하므로 처리 속도가 느리고 비용이 높다.

In [8]:
# !pip install langchain-experimental

# 시맨틱(Semantic) Chunking 예제 : 문장 간 의미 유사도 변화 지점을 기준으로 분할
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

text = """
LangChain은 LLM 애플리케이션 개발을 위한 프레임워크이다. 문서 로드, 텍스트 분할, 임베딩, 검색, 체인 등의 기능을 제공한다.
고양이는 혼자 있는 것을 좋아하는 반려동물이다. 강아지에 비해 독립적인 성향을 가지고 있다.
RAG 파이프라인의 성능은 Chunking 전략에 크게 좌우된다. 문맥이 유지되도록 분할해야 검색 정확도가 높아진다.
"""

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",   # percentile, standard_deviation, interquartile 중 선택
    breakpoint_threshold_amount=70,
)

semantic_docs = semantic_splitter.create_documents([text])

for i, doc in enumerate(semantic_docs):
    print(f"[시맨틱] 청크 {i+1}:", doc.page_content)

[시맨틱] 청크 1: 
LangChain은 LLM 애플리케이션 개발을 위한 프레임워크이다. 문서 로드, 텍스트 분할, 임베딩, 검색, 체인 등의 기능을 제공한다. 고양이는 혼자 있는 것을 좋아하는 반려동물이다.
[시맨틱] 청크 2: 강아지에 비해 독립적인 성향을 가지고 있다. RAG 파이프라인의 성능은 Chunking 전략에 크게 좌우된다. 문맥이 유지되도록 분할해야 검색 정확도가 높아진다.
[시맨틱] 청크 3: 


In [24]:
# [원인] breakpoint_threshold_type="gradient" 인데 breakpoint_threshold_amount를 지정하지 않음
# -> gradient 방식의 기본 임계값(BREAKPOINT_DEFAULTS)은 95(퍼센타일)이다.
# -> 문장이 16개뿐이라 distance-gradient 값도 15개뿐인데, "상위 5%"에 해당하는 지점은 사실상 문서 맨 끝 1곳뿐이다.
# -> 그래서 LangChain/RAG/VectorDB/피카소/태양 5개 주제가 뚜렷이 나뉘어 있어도 분할점을 못 찾고 전부 한 청크로 뭉쳐버린다.
# -> 즉 버그가 아니라 "임계값(threshold)을 너무 엄격하게" 잡아서 생긴 문제이다.

from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

text = """
LangChain은 LLM 애플리케이션 개발을 위한 프레임워크이다.
Prompt, Memory, Agent 등을 제공한다.
다양한 LLM과 쉽게 연동할 수 있다.

RAG는 외부 문서를 검색하여 답변의 정확도를 높인다.
Vector Database를 이용하여 관련 문서를 검색한다.
검색된 문서를 LLM의 Context로 전달한다.

벡터DB에는 Qdrant, Pinecone, Weaviate 등이 있다.
Embedding Vector를 저장한다.
유사도 검색을 수행한다.

피카소는 입체파 미술을 대표하는 화가이다.
대표작으로 게르니카가 있다.
20세기 현대 미술에 큰 영향을 주었다.

태양은 태양계의 중심에 위치한 항성이다.
지구는 태양을 약 365일 동안 공전한다.
빛이 지구에 도달하는 데 약 8분이 걸린다.

"""

# [수정] breakpoint_threshold_type="percentile" + breakpoint_threshold_amount를 낮게(70~80) 명시
#        -> 문장 간 거리(distance) 상위 20~30% 지점을 분할 기준으로 삼아 주제가 바뀌는 지점을 더 잘 잡아낸다.
splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=75,
)
chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}")
    print(chunk)
    print("=" * 50)

# 참고 : 몇 개의 청크로 나눌지 미리 알고 있다면 number_of_chunks로 직접 지정할 수도 있다.
# splitter = SemanticChunker(embeddings, number_of_chunks=5)

Chunk 1

LangChain은 LLM 애플리케이션 개발을 위한 프레임워크이다. Prompt, Memory, Agent 등을 제공한다. 다양한 LLM과 쉽게 연동할 수 있다. RAG는 외부 문서를 검색하여 답변의 정확도를 높인다. Vector Database를 이용하여 관련 문서를 검색한다.
Chunk 2
검색된 문서를 LLM의 Context로 전달한다. 벡터DB에는 Qdrant, Pinecone, Weaviate 등이 있다. Embedding Vector를 저장한다.
Chunk 3
유사도 검색을 수행한다.
Chunk 4
피카소는 입체파 미술을 대표하는 화가이다. 대표작으로 게르니카가 있다.
Chunk 5
20세기 현대 미술에 큰 영향을 주었다. 태양은 태양계의 중심에 위치한 항성이다. 지구는 태양을 약 365일 동안 공전한다. 빛이 지구에 도달하는 데 약 8분이 걸린다. 


In [27]:
# 몇개의 청크로 나눌지 미리 알고 있는 경우
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

text = """
LangChain은 LLM 애플리케이션 개발을 위한 프레임워크이다.
Prompt, Memory, Agent 등을 제공한다.
다양한 LLM과 쉽게 연동할 수 있다.

RAG는 외부 문서를 검색하여 답변의 정확도를 높인다.
Vector Database를 이용하여 관련 문서를 검색한다.
검색된 문서를 LLM의 Context로 전달한다.

벡터DB에는 Qdrant, Pinecone, Weaviate 등이 있다.
Embedding Vector를 저장한다.
유사도 검색을 수행한다.

피카소는 입체파 미술을 대표하는 화가이다.
대표작으로 게르니카가 있다.
20세기 현대 미술에 큰 영향을 주었다.

태양은 태양계의 중심에 위치한 항성이다.
지구는 태양을 약 365일 동안 공전한다.
빛이 지구에 도달하는 데 약 8분이 걸린다.

"""

# 몇 개의 청크로 나눌지 미리 알고 있다면 number_of_chunks로 직접 지정할 수도 있다.
# 2~5까지 변경해본다
splitter = SemanticChunker(embeddings, number_of_chunks=5)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}")
    print(chunk)
    print("=" * 50)

Chunk 1

LangChain은 LLM 애플리케이션 개발을 위한 프레임워크이다. Prompt, Memory, Agent 등을 제공한다. 다양한 LLM과 쉽게 연동할 수 있다. RAG는 외부 문서를 검색하여 답변의 정확도를 높인다. Vector Database를 이용하여 관련 문서를 검색한다.
Chunk 2
검색된 문서를 LLM의 Context로 전달한다. 벡터DB에는 Qdrant, Pinecone, Weaviate 등이 있다. Embedding Vector를 저장한다.
Chunk 3
유사도 검색을 수행한다.
Chunk 4
피카소는 입체파 미술을 대표하는 화가이다. 대표작으로 게르니카가 있다.
Chunk 5
20세기 현대 미술에 큰 영향을 주었다. 태양은 태양계의 중심에 위치한 항성이다. 지구는 태양을 약 365일 동안 공전한다. 빛이 지구에 도달하는 데 약 8분이 걸린다. 


### [6] 계층적 Chunking (Hierarchical Chunking)
: 문서의 구조(장 → 절 → 문단)를 먼저 인식하여 분할한 뒤, 필요하면 각 구조 내부를 다시 고정 크기로 세분화하는 방식이다. <br>
MarkdownHeaderTextSplitter 로 헤더(#, ##) 기준 1차 분할 후, RecursiveCharacterTextSplitter 로 2차 분할하면 구조 정보(metadata)를 유지한 채 세밀한 청크를 만들 수 있다. <br>
문맥 유지와 검색 정확도가 가장 뛰어나 실무에서 가장 추천되는 방식이다.

In [28]:
# 계층적(Hierarchical) Chunking 예제 : 문서 구조(장→절) 1차 분할 + 고정 크기 2차 분할
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

markdown_text = """
# 1장. LangChain 소개
LangChain은 LLM 애플리케이션 개발을 위한 프레임워크이다.

## 1.1 핵심 구성 요소
체인, 메모리, 에이전트, 리트리버 등이 있다.

## 1.2 활용 분야
RAG, 챗봇, 에이전트 자동화 등에 사용된다.

# 2장. 텍스트 분할 전략
문서를 적절한 크기의 청크로 나누는 것은 RAG 성능에 큰 영향을 준다.

## 2.1 고정 분할
문자 수 또는 토큰 수 기준으로 단순하게 나눈다.

## 2.2 시맨틱 분할
문장의 의미 유사도를 기준으로 나눈다.
"""

# 1단계 : 헤더(#, ##) 기준으로 장/절 구조를 유지하며 1차 분할
headers_to_split_on = [
    ("#", "chapter"),
    ("##", "section"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
header_splits = markdown_splitter.split_text(markdown_text)

# 2단계 : 각 절(section) 내부를 다시 고정 크기로 세분화 (metadata는 그대로 유지)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
hierarchical_chunks = child_splitter.split_documents(header_splits)

for i, doc in enumerate(hierarchical_chunks):
    print(f"[계층적] 청크 {i+1} {doc.metadata} :", doc.page_content)

[계층적] 청크 1 {'chapter': '1장. LangChain 소개'} : LangChain은 LLM 애플리케이션 개발을 위한 프레임워크이다.
[계층적] 청크 2 {'chapter': '1장. LangChain 소개', 'section': '1.1 핵심 구성 요소'} : 체인, 메모리, 에이전트, 리트리버 등이 있다.
[계층적] 청크 3 {'chapter': '1장. LangChain 소개', 'section': '1.2 활용 분야'} : RAG, 챗봇, 에이전트 자동화 등에 사용된다.
[계층적] 청크 4 {'chapter': '2장. 텍스트 분할 전략'} : 문서를 적절한 크기의 청크로 나누는 것은 RAG 성능에 큰 영향을 준다.
[계층적] 청크 5 {'chapter': '2장. 텍스트 분할 전략', 'section': '2.1 고정 분할'} : 문자 수 또는 토큰 수 기준으로 단순하게 나눈다.
[계층적] 청크 6 {'chapter': '2장. 텍스트 분할 전략', 'section': '2.2 시맨틱 분할'} : 문장의 의미 유사도를 기준으로 나눈다.


### 정리
- 빠르게 프로토타이핑하거나 문서 형식이 단순하다면 **고정 Chunking** 으로 충분하다. <br>
- 검색 정확도가 중요한 RAG 서비스라면 **시맨틱 Chunking** 을 고려한다. <br>
- 보고서, 매뉴얼처럼 장/절 구조가 뚜렷한 문서라면 **계층적 Chunking** 이 가장 좋은 결과를 낸다. <br>
- 실무에서는 계층적 분할로 큰 틀을 잡고, 내부는 고정/시맨틱 분할을 함께 적용하는 **하이브리드 방식**도 널리 쓰인다.

## [실무 정리] 가장 추천하는 Chunking 전략
결론부터 : 실무에서는 **"계층적(Hierarchical) 1차 분할 + 고정(Fixed) 크기 2차 분할" 하이브리드**를 기본값으로 쓰고, <br>
검색 정확도가 특히 중요한 구간에만 **시맨틱(Semantic) 청킹을 보완적으로** 적용하는 것이 가장 추천된다. <br>
**Header + Recursive Chunking** ★★★★★ (PDF에 가장 추천)

### 1) 문서 유형별 추천

| 문서 유형 | 추천 전략 | 이유 |
| --- | --- | --- |
| 매뉴얼 / 보고서 / 사내 위키 (제목 구조가 뚜렷) | 계층적(Hierarchical) | 장/절 metadata로 출처 추적 가능, 검색 정확도 최상 |
| 단순 텍스트 / 로그 / 자막처럼 구조가 없는 문서 | 고정(Fixed) | 계층이 없어 구조 분할의 이점이 없음, 가장 빠르고 저렴 |
| FAQ / 상담 이력처럼 주제가 자주 바뀌는 문서 | 시맨틱(Semantic) | 문장 단위로 주제 전환 지점을 잡아야 문맥 손실이 적음 |
| 대규모 문서 + 비용/속도가 민감한 경우 | 고정으로 우선 처리 후, 중요 구간만 시맨틱 재분할 | 전체를 시맨틱으로 돌리면 임베딩 호출이 급증해 비용/지연시간이 커짐 |

### 2) 실무 표준 파이프라인 (권장 기본값)
1. **1차 분할(구조 인식)** : `MarkdownHeaderTextSplitter` / `HTMLHeaderTextSplitter` 등으로 장(chapter)·절(section) 구조를 먼저 나눈다. <br>
2. **2차 분할(크기 보정)** : `RecursiveCharacterTextSplitter(chunk_size=300~500자, chunk_overlap=10~20%)`로 절 내부를 다시 균일한 크기로 세분화한다. <br>
3. **(옵션) 시맨틱 보완** : 검색 정확도가 특히 중요한 컬렉션에 한해, 2차 결과를 `SemanticChunker`로 재병합/재분할한다.

### 3) 시맨틱 Chunking 적용 시 주의점 (앞선 실습에서 직접 확인한 문제 기반)
- `breakpoint_threshold_amount` 를 반드시 명시적으로 지정할 것. 기본값(95퍼센타일)은 너무 엄격해서 문장 수가 적은 문서에서는 분할이 거의 일어나지 않을 수 있다. <br>
- 몇 개의 청크로 나눌지 미리 알고 있다면 `breakpoint_threshold_amount` 튜닝보다 `number_of_chunks` 를 직접 지정하는 편이 더 안정적이다. <br>
- 문장 수가 적을수록 percentile/gradient 계산이 불안정해지므로, 운영 전에는 항상 실제 출력 청크를 눈으로 검증해야 한다. <br>
- 문장 단위 임베딩 호출이 필요해 비용·지연시간이 고정 방식보다 훨씬 크다. 전체 파이프라인이 아니라 필요한 구간에만 선택적으로 적용한다.

### 4) 최종 결론
| 구분 | 고정 Chunking (Fixed) | 시맨틱 Chunking (Semantic) | 계층적 Chunking (Hierarchical) |
| --- | --- | --- | --- |
| 실무 기본 채택 여부 | 보조 수단(2차 분할용) | 선택적 보완 | **기본값(1차 분할)** |
| 추천도 | ★★★☆☆ | ★★★★☆ | ★★★★★ |

**실무 기본값 = 계층적 + 고정 하이브리드.** 검색 품질이 핵심 KPI인 프로덕션 RAG라면 여기에 시맨틱 청킹을 보완적으로 추가한다.